# Initial EDA

This notebook explores the raw CSV files before building the dbt models. The goal is to understand the table structure, key fields, relationships, date quality, and basic data quality issues that should be handled later in staging.

### 1. Load Raw Data
First, I load all raw CSV files from the `data/raw` folder.

In [42]:
from pathlib import Path
import pandas as pd

In [61]:
raw_path = Path("../data/raw")

tables = {
    file.stem: pd.read_csv(file)
    for file in raw_path.glob("*.csv")
}

sorted(tables.keys())

['raw_campaigns',
 'raw_clients',
 'raw_pos',
 'raw_projects',
 'raw_questions',
 'raw_responses',
 'raw_route_employee',
 'raw_routes',
 'raw_visits',
 'raw_workers']

### 2. Table Overview
This gives a first overview of the dataset: number of rows, number of columns, and full duplicate rows in each raw table.

In [75]:
overview = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum()
    }
    for name, df in tables.items()
])

overview

,table,rows,columns,duplicate_rows
0,raw_campaigns,72,16,0
1,raw_clients,15,7,1
2,raw_pos,250,12,0
3,raw_projects,28,8,0
4,raw_questions,648,12,0
5,raw_responses,10192,10,0
6,raw_routes,241,12,0
7,raw_route_employee,482,8,0
8,raw_visits,1762,15,15
9,raw_workers,47,9,2


### 3. Primary Key Checks
I check whether the expected primary key columns are non-null and unique. This is important because duplicate or null primary keys can break joins and cause double counting in the final metrics.

In [45]:
primary_keys = {
    "raw_clients": "client_id",
    "raw_projects": "project_id",
    "raw_campaigns": "campaign_id",
    "raw_questions": "question_id",
    "raw_workers": "employee_id",
    "raw_pos": "intervention_point_id",
    "raw_routes": "route_id",
    "raw_route_employee": "route_employee_id",
    "raw_visits": "visit_id",
    "raw_responses": "answer_id",
}

pk_check = pd.DataFrame([
    {
        "table": table,
        "primary_key": pk,
        "rows": len(tables[table]),
        "null_keys": tables[table][pk].isna().sum(),
        "unique_keys": tables[table][pk].nunique(),
        "duplicate_keys": len(tables[table]) - tables[table][pk].nunique()
    }
    for table, pk in primary_keys.items()
])

pk_check

,table,primary_key,rows,null_keys,unique_keys,duplicate_keys
0,raw_clients,client_id,15,0,14,1
1,raw_projects,project_id,28,0,28,0
2,raw_campaigns,campaign_id,72,0,72,0
3,raw_questions,question_id,648,0,648,0
4,raw_workers,employee_id,47,0,45,2
5,raw_pos,intervention_point_id,250,0,250,0
6,raw_routes,route_id,241,0,241,0
7,raw_route_employee,route_employee_id,482,0,482,0
8,raw_visits,visit_id,1762,0,1747,15
9,raw_responses,answer_id,10192,0,10192,0


### 4. Duplicate Key Sample Check

The primary key check shows the count of duplicate keys, but I also inspect sample duplicate rows to understand whether they are exact duplicates or conflicting records.

In [46]:
for table, pk in {
    "raw_clients": "client_id",
    "raw_workers": "employee_id",
    "raw_visits": "visit_id"
}.items():
    duplicates = tables[table][
        tables[table][pk].duplicated(keep=False)
    ].sort_values(pk)

    print(f"\n{table} duplicate {pk} rows:")
    display(duplicates.head(10))


raw_clients duplicate client_id rows:


,client_id,client_name,company_id,country,sector,created_at,updated_at_sys
0,1,Laboratorios Valtera España,1,spain,Farma,2021-10-17,2023-06-13
14,1,Laboratorios Valtera España,1,spain,Farma,2021-10-17,2023-06-13



raw_workers duplicate employee_id rows:


,employee_id,employee_first_name,employee_active_status,employee_hire_date,employee_address_province,employee_contract_type,company_id,created_at,updated_at_sys
5,6,Carlos,False,2018-08-09,TARRAGONA,NaN,1,2018-08-09,2022-08-04
45,6,Carlos,False,2018-08-09,TARRAGONA,NaN,1,2018-08-09,2022-08-04
10,11,Pablo,True,2024-10-11,Hauts-de-Seine,PART,1,2024-10-11,2024-12-22
46,11,Pablo,True,2024-10-11,Hauts-de-Seine,PART,1,2024-10-11,2024-12-22



raw_visits duplicate visit_id rows:


,visit_id,campaign_id,campaign_code,project_code,intervention_point_id,intervention_point_code,route_id,route_code,visit_date,visit_time,visit_status,visit_type,is_client_billable,created_at,updated_at_sys
19,20,2,CAM00002,PRJ0001,195,PDV000195,7.0,RUT00007,2026-01-24,08:45:00,INCID,T,NaN,2026-01-21,2026-04-18
1749,20,2,CAM00002,PRJ0001,195,PDV000195,7.0,RUT00007,2026-01-24,08:45:00,INCID,T,NaN,2026-01-21,2026-04-18
1758,124,5,CAM00005,PRJ0003,147,NaN,17.0,NaN,2026-02-04,13:00:00,INCID,C,True,2026-02-04,2026-02-11
123,124,5,CAM00005,PRJ0003,147,NaN,17.0,NaN,2026-02-04,13:00:00,INCID,C,True,2026-02-04,2026-02-11
157,158,7,CAM00007,PRJ0005,134,PDV000134,21.0,RUT00021,2026-01-06,11:30:00,INCID,C,True,2026-01-01,2026-04-16
1756,158,7,CAM00007,PRJ0005,134,PDV000134,21.0,RUT00021,2026-01-06,11:30:00,INCID,C,True,2026-01-01,2026-04-16
1753,159,7,CAM00007,PRJ0005,148,PDV000148,23.0,RUT00023,2026-01-29,15:15:00,NOVIS,T,True,2026-01-28,2026-02-07
158,159,7,CAM00007,PRJ0005,148,PDV000148,23.0,RUT00023,2026-01-29,15:15:00,NOVIS,T,True,2026-01-28,2026-02-07
170,171,7,CAM00007,PRJ0005,226,PDV000226,22.0,RUT00022,2026-02-08,15:45:00,INCID,NaN,False,2026-02-04,2026-02-17
1759,171,7,CAM00007,PRJ0005,226,PDV000226,22.0,RUT00022,2026-02-08,15:45:00,INCID,NaN,False,2026-02-04,2026-02-17


### 5. Foreign Key Checks
I check the most important relationships between tables. They are documented because they can affect joins and dashboard metrics.

In [47]:
def orphan_check(child_table, child_key, parent_table, parent_key):
    child = tables[child_table]
    parent = tables[parent_table]

    orphan_rows = child[
        child[child_key].notna()
        & ~child[child_key].isin(parent[parent_key])
    ]

    return {
        "relationship": f"{child_table}.{child_key} → {parent_table}.{parent_key}",
        "orphan_rows": len(orphan_rows)
    }

In [48]:
fk_check = pd.DataFrame([
    orphan_check("raw_projects", "client_id", "raw_clients", "client_id"),
    orphan_check("raw_campaigns", "project_id", "raw_projects", "project_id"),
    orphan_check("raw_questions", "campaign_id", "raw_campaigns", "campaign_id"),
    orphan_check("raw_routes", "campaign_id", "raw_campaigns", "campaign_id"),
    orphan_check("raw_visits", "campaign_id", "raw_campaigns", "campaign_id"),
    orphan_check("raw_visits", "route_id", "raw_routes", "route_id"),
    orphan_check("raw_visits", "intervention_point_id", "raw_pos", "intervention_point_id"),
    orphan_check("raw_responses", "visit_id", "raw_visits", "visit_id"),
    orphan_check("raw_responses", "question_id", "raw_questions", "question_id"),
])

fk_check

,relationship,orphan_rows
0,raw_projects.client_id → raw_clients.client_id,0
1,raw_campaigns.project_id → raw_projects.projec...,0
2,raw_questions.campaign_id → raw_campaigns.camp...,0
3,raw_routes.campaign_id → raw_campaigns.campaig...,0
4,raw_visits.campaign_id → raw_campaigns.campaig...,0
5,raw_visits.route_id → raw_routes.route_id,70
6,raw_visits.intervention_point_id → raw_pos.int...,0
7,raw_responses.visit_id → raw_visits.visit_id,0
8,raw_responses.question_id → raw_questions.ques...,0


### 6. Orphan Route Inspection

The foreign key check shows that some visits reference route IDs that do not exist in `raw_routes`. I inspect those route IDs to understand whether this looks like random bad data or a consistent pattern.

In [50]:
orphan_route_visits = tables["raw_visits"][
    tables["raw_visits"]["route_id"].notna()
    & ~tables["raw_visits"]["route_id"].isin(tables["raw_routes"]["route_id"])
]

orphan_route_visits["route_id"].describe()

count       70.000000
mean     94690.671429
std       2910.718970
min      90052.000000
25%      92148.000000
50%      94693.000000
75%      97122.000000
max      99970.000000
Name: route_id, dtype: float64

In [51]:
orphan_route_visits["route_id"].value_counts().head(10)

route_id
99970.0    1
97041.0    1
90558.0    1
98901.0    1
97844.0    1
98057.0    1
96236.0    1
92284.0    1
96122.0    1
91202.0    1
Name: count, dtype: int64

## 7. Status and Type Checks

I check important categorical fields that will be used later in metrics, filters, and dbt accepted value tests.

In [52]:
tables["raw_visits"]["visit_status"].value_counts(dropna=False)

visit_status
INCID    1187
INFO      279
NOVIS     138
OK         87
NaN        71
Name: count, dtype: int64

In [53]:
tables["raw_questions"]["question_type"].value_counts(dropna=False)

question_type
S      287
T      124
M      116
N       93
NaN     28
Name: count, dtype: int64

In [54]:
tables["raw_campaigns"]["campaign_state"].value_counts(dropna=False)

campaign_state
Activa            16
Pausada           16
En preparación    14
NaN               13
Cerrada           13
Name: count, dtype: int64

## 8. Location Field Standardisation Checks

I check descriptive location fields because they may be used later in PowerBI filters and geographical charts. 

In [76]:
location_checks = {
    "raw_clients.country": tables["raw_clients"]["country"].value_counts(dropna=False),
    "raw_pos.intervention_point_province": tables["raw_pos"]["intervention_point_province"].value_counts(dropna=False).head(20),
    "raw_workers.employee_address_province": tables["raw_workers"]["employee_address_province"].value_counts(dropna=False).head(20),
}

for field, counts in location_checks.items():
    print(f"\n{field}")
    display(counts)


raw_clients.country


country
España    7
spain     6
SPAIN     2
Name: count, dtype: int64


raw_pos.intervention_point_province


intervention_point_province
NaN                       14
Castellon                  9
CIUDAD REAL                8
TARRAGONA                  8
Barcelona                  7
Cáceres                    7
HUELVA                     7
Palmas, Las                6
Girona                     6
A Coruña                   6
MADRID                     6
SANTA CRUZ DE TENERIFE     6
Granada                    6
Asturias                   5
ZAMORA                     5
Lleida                     5
Pontevedra                 5
Zamora                     5
Segovia                    5
Andorra la Vella           5
Name: count, dtype: int64


raw_workers.employee_address_province


employee_address_province
Castellon                 3
Canillo                   2
TARRAGONA                 2
Castellón                 2
Cuenca                    2
ASTURIAS                  2
Hauts-de-Seine            2
Rioja, La                 2
Pontevedra                2
LUGO                      2
VALLADOLID                2
Escaldes-Engordany        2
Castelló                  1
Guarda                    1
MÁLAGA                    1
Santarém                  1
BADAJOZ                   1
NaN                       1
SANTA CRUZ DE TENERIFE    1
Portalegre                1
Name: count, dtype: int64

## 9. Date Format Checks

I check which columns contain dates and whether campaign dates use a consistent format.

Mixed date formats are important because automatic parsing can silently produce wrong dates.

In [55]:
for name, df in tables.items():
    date_cols = [col for col in df.columns if "date" in col.lower()]
    if date_cols:
        print(name, date_cols)

raw_campaigns ['campaign_start_date', 'campaign_end_date', 'updated_at_sys']
raw_clients ['updated_at_sys']
raw_pos ['updated_at']
raw_projects ['updated_at_sys']
raw_questions ['updated_at_sys']
raw_responses ['updated_at_sys']
raw_routes ['route_start_date', 'route_end_date', 'updated_at_sys']
raw_route_employee ['updated_at']
raw_visits ['visit_date', 'updated_at_sys']
raw_workers ['employee_hire_date', 'updated_at_sys']


In [56]:
campaigns = tables["raw_campaigns"]

date_patterns = pd.DataFrame({
    "campaign_start_iso": campaigns["campaign_start_date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$").sum(),
    "campaign_start_eu": campaigns["campaign_start_date"].astype(str).str.match(r"^\d{2}/\d{2}/\d{4}$").sum(),
    "campaign_end_iso": campaigns["campaign_end_date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$").sum(),
    "campaign_end_eu": campaigns["campaign_end_date"].astype(str).str.match(r"^\d{2}/\d{2}/\d{4}$").sum(),
}, index=[0])

date_patterns

,campaign_start_iso,campaign_start_eu,campaign_end_iso,campaign_end_eu
0,60,12,60,12


## 10. Campaign Date Validation

Because campaign dates use both ISO and DD/MM/YYYY formats, I parse the two known formats explicitly instead of relying on automatic date inference. Then I check whether any campaigns have an end date earlier than their start date.

In [57]:
def parse_campaign_date(series):
    s = series.astype("string").str.strip()

    parsed_iso = pd.to_datetime(
        s,
        format="%Y-%m-%d",
        errors="coerce"
    )

    parsed_eu = pd.to_datetime(
        s,
        format="%d/%m/%Y",
        errors="coerce"
    )

    return parsed_iso.fillna(parsed_eu)


campaigns = tables["raw_campaigns"].copy()

campaigns["start_date_parsed"] = parse_campaign_date(
    campaigns["campaign_start_date"]
)

campaigns["end_date_parsed"] = parse_campaign_date(
    campaigns["campaign_end_date"]
)

invalid_campaign_dates = campaigns[
    campaigns["end_date_parsed"] < campaigns["start_date_parsed"]
]

invalid_campaign_dates[
    ["campaign_id", "campaign_name", "campaign_start_date", "campaign_end_date"]
]

,campaign_id,campaign_name,campaign_start_date,campaign_end_date
27,28,Campaña Oct 2025 – Proyecto Mantenimiento 12,2025-10-05,2025-09-29
29,30,Campaña Oct 2025 – Proyecto Mantenimiento 13,2025-10-28,2025-10-23


## 11. Initial EDA Findings

The raw data is generally usable for modeling, but it needs some cleaning before building the dbt marts.

Main observations:

- The dataset has 10 raw CSV files covering clients, projects, campaigns, questions, workers, points of sale, routes, route-worker assignments, visits, and responses.
- `raw_responses` has 10,192 rows, while `raw_visits` has 1,762 rows and 1,747 unique `visit_id` values. This suggests that visits and responses should be modeled separately.
- Some primary keys are duplicated in `raw_clients`, `raw_workers`, and `raw_visits`.
- The main relationship issue is in `raw_visits.route_id`, where 70 visits reference routes that are not present in `raw_routes`.
- `visit_status` is missing in 71 rows. I would treat these as `UNKNOWN`, not `NOVIS`, because a missing status is not the same as a visit that was not carried out.
- `question_type` is missing in 28 rows, which may limit response analysis by question type.
- Campaign dates need special handling because they use mixed formats: 60 ISO-format values and 12 DD/MM/YYYY-format values for both start and end dates. Without explicit parsing, date filters and time-based analysis could be wrong.
- Two campaigns have an end date earlier than their start date, so they should be flagged in staging rather than silently corrected or dropped.
- Some location fields are not fully standardised. For example, country values appear with different casing or language variants, and province names include inconsistent casing, accents, or alternative spellings. This could affect geographical filters and charts if these fields are used for detailed regional reporting.

## 12. Main Data Quality Notes

These are the main issues I found during the initial EDA and how I plan to handle them later in dbt:

- Some primary keys are duplicated:
  - `raw_clients`: 1 duplicate `client_id`
  - `raw_workers`: 2 duplicate `employee_id` values
  - `raw_visits`: 15 duplicate `visit_id` values

  In staging, I would inspect these duplicates and deduplicate them using a documented rule. Where `updated_at_sys` is available, I would keep the latest record for each primary key.

- `raw_visits` has 70 rows where `route_id` does not exist in `raw_routes`. These route IDs are outside the normal route ID range.

  I would keep these visits because they may represent real field activity. Instead of dropping them, I would add a flag such as `is_orphan_route`.

- `visit_status` is missing in 71 visit rows.

  I would standardize these values as `UNKNOWN`, not `NOVIS`. `NOVIS` means the visit was not carried out, while a null value means the status is unknown or missing.

- `question_type` is missing in 28 question rows.

  I would standardize missing question types as `UNKNOWN` so these questions are still visible in response analysis.

- Campaign dates are not stored in one consistent format. There are 60 ISO-format values and 12 DD/MM/YYYY-format values for both campaign start and end dates.

  I would parse the known formats explicitly in staging. Without this, date filters and time-based metrics could be wrong.

- Two campaigns have an end date earlier than their start date:
  - `campaign_id = 28`
  - `campaign_id = 30`

  I would keep these records but add a `has_invalid_date_range` flag.

- Some location fields need standardisation.

  The `country` field in `raw_clients` and province fields in `raw_pos` and `raw_workers` contain inconsistent labels. Examples include different casing, language variants, accents, or alternative spellings. I would not prioritise this over the main grain and relationship issues, but I would standardise these fields before using the dashboard for detailed geographical reporting.

- Worker first names are not unique.

  I would not use `employee_first_name` alone for worker level analysis or dashboard labels, because different workers can share the same first name. In PowerBI, I would use a label based on first name and `employee_id` together, such as `worker_label`, to avoid grouping different workers under the same name.

Most of these issues should be handled in the staging layer so the marts consume clean and standardised data. Issues related to analytical grain, such as duplicate visits and worker assignment, should also be handled carefully in the mart or PowerBI layer.